# Complete bases + detect wires (2 classes)

Runs AFTER `remove_trees.ipynb`. Two jobs:

1. **Extend each pole down to the ground** using the raw cloud. Fixes poles whose base got chopped off (the short-height ones — 9 of them have heights <7m when the median is 8.8m).
2. **Detect wires in two classes**: SAME_LANE (along each side of the road) and CROSS_ROAD (spanning across). KMeans auto-splits the 37 poles into left/right lane. Wire tracing uses catenary + linear fallback with cross-section width gate to reject any tree remnants.

**Inputs:** `poles_no_trees.las`, `pole_locations_no_trees.csv`, `laser_left.las`

**Outputs:** `poles_final.las` (all pole + wire points, 3 classes), `poles_final.qml`, `wires.csv`

In [1]:
import numpy as np
import pandas as pd
import laspy
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans, DBSCAN

# --- paths ---
RAW_LAS   = 'laser_left.las'
POLE_LAS  = 'poles_no_trees.las'
POLE_CSV  = 'pole_locations_no_trees.csv'
OUT_LAS   = 'poles_final.las'
OUT_QML   = 'poles_final.qml'
OUT_WIRES = 'wires.csv'

# --- load ---
print('Loading raw cloud...')
raw = laspy.read(RAW_LAS)
raw_pts = np.vstack([raw.x, raw.y, raw.z]).T
print(f'  {len(raw_pts):,} raw points')

print('Loading filtered pole cloud...')
pl = laspy.read(POLE_LAS)
pole_pts = np.vstack([pl.x, pl.y, pl.z]).T
print(f'  {len(pole_pts):,} pole points')

poles = pd.read_csv(POLE_CSV)
print(f'  {len(poles)} pole structures')

raw_tree = cKDTree(raw_pts[:, :2])
pole_tree = cKDTree(pole_pts[:, :2])

Loading raw cloud...


  31,905,423 raw points
Loading filtered pole cloud...
  69,566 pole points
  37 pole structures


## Job 1 — Complete pole bases

For each pole:
1. Find the current base Z (min of existing pole points within 1.5m of center)
2. Estimate true ground Z from raw cloud (5th percentile of points within 2m XY around pole)
3. If current base is >1.5m above ground, pull raw points in the missing Z range within a 0.5m XY cylinder (tight to avoid bushes/ground clutter)
4. Add those as extension points to the pole cluster

In [2]:
EXTEND_RADIUS = 0.5     # tight cylinder for base extension
GROUND_RADIUS = 2.0     # wider radius to estimate ground Z
MIN_GAP_M     = 1.5     # only extend if gap to ground > this

extensions = []
extension_pid = []
extended_count = 0

for _, row in poles.iterrows():
    pid = int(row['Pole_ID'])
    px, py = float(row['X']), float(row['Y'])

    # Current pole points near this center
    my_idx = pole_tree.query_ball_point([px, py], r=1.5)
    if len(my_idx) < 10: continue
    current_base = float(pole_pts[my_idx, 2].min())

    # Ground Z from raw cloud in a 2m radius
    ground_idx = raw_tree.query_ball_point([px, py], r=GROUND_RADIUS)
    if len(ground_idx) < 20: continue
    ground_z = float(np.percentile(raw_pts[ground_idx, 2], 5))

    gap = current_base - ground_z
    if gap <= MIN_GAP_M: continue

    # Pull raw points in the missing Z range within a tight cylinder
    ext_idx = raw_tree.query_ball_point([px, py], r=EXTEND_RADIUS)
    if len(ext_idx) == 0: continue
    ext_pts_all = raw_pts[ext_idx]
    z_range = (ext_pts_all[:, 2] >= ground_z) & (ext_pts_all[:, 2] < current_base)
    ext_pts = ext_pts_all[z_range]
    if len(ext_pts) < 5: continue

    extensions.append(ext_pts)
    extension_pid.extend([pid] * len(ext_pts))
    extended_count += 1

if extensions:
    ext_pts_all = np.vstack(extensions)
    ext_pid_all = np.array(extension_pid, dtype=np.int32)
    print(f'Extended {extended_count} poles, adding {len(ext_pts_all):,} base points')
else:
    ext_pts_all = np.zeros((0, 3))
    ext_pid_all = np.zeros(0, dtype=np.int32)
    print('No poles needed base extension')

# Combine original pole points + extensions
# Assign pole_pts to nearest pole ID for the final output
centers = poles[['X','Y']].values
ctree = cKDTree(centers)
_, orig_nearest = ctree.query(pole_pts[:, :2], k=1)
orig_pid = poles['Pole_ID'].values[orig_nearest].astype(np.int32)

all_pole_pts = np.vstack([pole_pts, ext_pts_all])
all_pole_pid = np.concatenate([orig_pid, ext_pid_all])
print(f'Total pole points after extension: {len(all_pole_pts):,}')

Extended 13 poles, adding 9,528 base points
Total pole points after extension: 79,094


## Job 2a — Classify poles into left / right lane

KMeans on X coordinate finds the two lanes automatically. Prints the split so you can verify it looks right.

In [3]:
# Compute robust pole tops from the RAW cloud (98th pct within 1.2m) — for accurate wire Z
pole_tops = {}
for _, row in poles.iterrows():
    pid = int(row['Pole_ID'])
    px, py = float(row['X']), float(row['Y'])
    idxs = raw_tree.query_ball_point([px, py], r=1.2)
    if len(idxs) >= 10:
        top_z = float(np.percentile(raw_pts[idxs, 2], 98))
    else:
        top_z = float(row['Z'])
    pole_tops[pid] = (px, py, top_z)

# KMeans on X → two lanes
X = poles[['X']].values
km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(X)
lane_labels = km.labels_
# Label 0 = west (lower X), label 1 = east (higher X) — reorder to make sure
if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0]:
    lane_labels = 1 - lane_labels
poles['Lane'] = lane_labels  # 0 = west, 1 = east

print(f"Lane split at X ≈ {km.cluster_centers_.mean():.1f}")
print(f"  West lane: {(lane_labels == 0).sum()} poles")
print(f"  East lane: {(lane_labels == 1).sum()} poles")

# Per-pole lookup
pole_lane = {int(r['Pole_ID']): int(r['Lane']) for _, r in poles.iterrows()}

Lane split at X ≈ 356470.9
  West lane: 19 poles
  East lane: 18 poles


## Job 2b — Detect wires with two classes

For every pole pair within span range:
- Same lane → `SAME_LANE_WIRE`
- Different lanes → `CROSS_ROAD_WIRE`

Three-pass detection per pair: strict catenary, loose catenary, linear tube. Cross-section width gate (<0.4m perpendicular spread) rejects tree remnants.

In [4]:
MIN_SPAN, MAX_SPAN = 5.0, 90.0
TUBE_A, COV_A, CAT_A = 1.8, 0.10, 1.0
TUBE_B, COV_B, CAT_B = 2.5, 0.05, 1.5
TUBE_C, Z_LINE_C     = 2.5, 1.5
Z_BELOW, Z_ABOVE     = 3.5, 1.5
Z_BAND_EPS           = 0.5
MAX_ACROSS_STD       = 0.4       # tree rejection gate

pole_ids  = poles['Pole_ID'].values.astype(int)
pole_xys  = poles[['X','Y']].values

# Enumerate all pairs within MAX_SPAN
pairs = []
for i in range(len(pole_xys)):
    for j in range(i+1, len(pole_xys)):
        d = float(np.linalg.norm(pole_xys[i] - pole_xys[j]))
        if MIN_SPAN < d < MAX_SPAN:
            pairs.append((i, j, d))
print(f'Candidate pairs within {MIN_SPAN}-{MAX_SPAN}m: {len(pairs)}')

def fit_parabola(s, z):
    A_ = np.stack([np.ones_like(s), s, s*s], axis=1)
    return np.linalg.lstsq(A_, z, rcond=None)[0]

def trace_catenary(i, j, tube_w, min_cov, cat_r):
    A = pole_xys[i]; B = pole_xys[j]
    pa, pb = int(pole_ids[i]), int(pole_ids[j])
    aZ, bZ = pole_tops[pa][2], pole_tops[pb][2]
    vec = B - A; span = float(np.linalg.norm(vec))
    u = vec / span; n = np.array([-u[1], u[0]])
    cand = np.array(raw_tree.query_ball_point((A+B)/2, r=span/2 + tube_w + 2), dtype=np.int64)
    if len(cand) == 0: return []
    cpts = raw_pts[cand]
    rel = cpts[:, :2] - A
    along = rel @ u; across = rel @ n
    low, high = min(aZ, bZ), max(aZ, bZ)
    init = ((along > 0.5) & (along < span-0.5) & (np.abs(across) < tube_w) &
            (cpts[:, 2] > low - Z_BELOW) & (cpts[:, 2] < high + Z_ABOVE))
    if init.sum() < 4: return []
    walong = along[init]
    if (walong.max() - walong.min())/span < min_cov: return []
    wpts = cpts[init]; wacr = across[init]
    z_dev = wpts[:, 2] - (aZ*(1-walong/span) + bZ*(walong/span))
    db = DBSCAN(eps=Z_BAND_EPS, min_samples=3).fit(z_dev.reshape(-1,1))
    out = []
    for lbl in set(db.labels_):
        if lbl == -1: continue
        b = db.labels_ == lbl
        if b.sum() < 3: continue
        if np.std(wacr[b]) > MAX_ACROSS_STD: continue        # tree gate
        try: coef = fit_parabola(walong[b], wpts[b, 2])
        except Exception: continue
        in_tube = (along > 0.5) & (along < span-0.5) & (np.abs(across) < tube_w)
        z_pred = coef[0] + coef[1]*along + coef[2]*along*along
        near = np.abs(cpts[:, 2] - z_pred) < cat_r
        fm = in_tube & near
        if fm.sum() < 4: continue
        if np.std(across[fm]) > MAX_ACROSS_STD: continue     # tree gate 2
        ra = along[fm]
        if (ra.max() - ra.min())/span < min_cov: continue
        out.extend(int(g) for g in cand[fm])
    return out

def trace_linear(i, j):
    A = pole_xys[i]; B = pole_xys[j]
    pa, pb = int(pole_ids[i]), int(pole_ids[j])
    aZ, bZ = pole_tops[pa][2], pole_tops[pb][2]
    vec = B - A; span = float(np.linalg.norm(vec))
    u = vec / span; n = np.array([-u[1], u[0]])
    cand = np.array(raw_tree.query_ball_point((A+B)/2, r=span/2 + TUBE_C + 2), dtype=np.int64)
    if len(cand) == 0: return []
    cpts = raw_pts[cand]
    rel = cpts[:, :2] - A
    along = rel @ u; across = rel @ n
    line_z = aZ + (bZ - aZ) * (along/span)
    mask = ((along > 0.3) & (along < span-0.3) &
            (np.abs(across) < TUBE_C) &
            (np.abs(cpts[:, 2] - line_z) < Z_LINE_C) &
            (cpts[:, 2] > min(aZ, bZ) - 3.5))
    if mask.sum() < 5: return []
    if np.std(across[mask]) > MAX_ACROSS_STD: return []
    if (along[mask].max() - along[mask].min())/span < 0.05: return []
    return [int(g) for g in cand[mask]]

# Run all pairs
wire_assignments = {}   # global_raw_idx -> wire_class (70=SAME_LANE, 71=CROSS_ROAD)
span_rows = []

for (i, j, d) in pairs:
    pa, pb = int(pole_ids[i]), int(pole_ids[j])
    cross = pole_lane[pa] != pole_lane[pb]
    cls = 71 if cross else 70
    kind = 'CROSS_ROAD' if cross else 'SAME_LANE'

    # Try A → B → C
    pts_idx = trace_catenary(i, j, TUBE_A, COV_A, CAT_A)
    pass_used = 'A' if pts_idx else None
    if not pts_idx:
        pts_idx = trace_catenary(i, j, TUBE_B, COV_B, CAT_B)
        if pts_idx: pass_used = 'B'
    if not pts_idx:
        pts_idx = trace_linear(i, j)
        if pts_idx: pass_used = 'C'

    if pts_idx:
        for g in pts_idx:
            if g not in wire_assignments:
                wire_assignments[g] = cls
        span_rows.append({'pole_a': pa, 'pole_b': pb, 'span_m': round(d, 1),
                          'kind': kind, 'pass': pass_used, 'points': len(pts_idx)})

spans_df = pd.DataFrame(span_rows).sort_values(['kind','pole_a','pole_b'])
spans_df.to_csv(OUT_WIRES, index=False)

print(f'\nSpans detected: {len(spans_df)}')
if len(spans_df):
    print(f"  SAME_LANE:  {(spans_df['kind']=='SAME_LANE').sum()}")
    print(f"  CROSS_ROAD: {(spans_df['kind']=='CROSS_ROAD').sum()}")
    print(f"\nBy pass:")
    for p in ['A','B','C']:
        print(f"  Pass {p}: {(spans_df['pass']==p).sum()}")
print(f'\nTotal wire points: {len(wire_assignments):,}')
print(f'{OUT_WIRES} written')

Candidate pairs within 5.0-90.0m: 103



Spans detected: 28
  SAME_LANE:  24
  CROSS_ROAD: 4

By pass:
  Pass A: 24
  Pass B: 3
  Pass C: 1

Total wire points: 5,285
wires.csv written


## Save final LAS + QML

Three classes total:
- **65 POLE** (green) — all pole points, including completed bases
- **70 SAME_LANE_WIRE** (blue)
- **71 CROSS_ROAD_WIRE** (red)

In [5]:
# Assemble final point set
if wire_assignments:
    wg = np.array(list(wire_assignments.keys()), dtype=np.int64)
    wire_pts = raw_pts[wg]
    wire_cls = np.array([wire_assignments[int(g)] for g in wg], dtype=np.uint8)
else:
    wire_pts = np.zeros((0, 3))
    wire_cls = np.zeros(0, dtype=np.uint8)

final_pts = np.vstack([all_pole_pts, wire_pts])
final_cls = np.concatenate([
    np.full(len(all_pole_pts), 65, dtype=np.uint8),
    wire_cls
])
final_pid = np.clip(np.concatenate([
    all_pole_pid,
    np.zeros(len(wire_pts), dtype=np.int32),
]) + 1, 0, 255).astype(np.uint8)

print(f'Writing {len(final_pts):,} points total')
print(f"  POLE:           {(final_cls == 65).sum():>7,}")
print(f"  SAME_LANE_WIRE: {(final_cls == 70).sum():>7,}")
print(f"  CROSS_ROAD_WIRE:{(final_cls == 71).sum():>7,}")

header = laspy.LasHeader(point_format=raw.header.point_format, version=raw.header.version)
header.scales  = raw.header.scales
header.offsets = raw.header.offsets
out = laspy.LasData(header)
out.x, out.y, out.z = final_pts.T
out.classification = final_cls
out.user_data = final_pid
out.write(OUT_LAS)
print(f'{OUT_LAS} written')

# QML with labeled classes
qml = '''<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis styleCategories="Symbology" version="3.34">
  <renderer-v2 type="pointcloudclassified" attribute="Classification">
    <categories>
      <category value="65" label="POLE"            color="60,200,60,255"    render="true"/>
      <category value="70" label="SAME_LANE_WIRE"  color="60,130,220,255"   render="true"/>
      <category value="71" label="CROSS_ROAD_WIRE" color="220,60,60,255"    render="true"/>
    </categories>
  </renderer-v2>
</qgis>
'''
with open(OUT_QML, 'w') as f: f.write(qml)
print(f'{OUT_QML} written')
print('\nDone. In QGIS: Layer Properties → Symbology → Load Style → poles_final.qml')

Writing 84,379 points total
  POLE:            79,094
  SAME_LANE_WIRE:   3,860
  CROSS_ROAD_WIRE:  1,425
poles_final.las written
poles_final.qml written

Done. In QGIS: Layer Properties → Symbology → Load Style → poles_final.qml


## Done

Open `poles_final.las` in QGIS and load `poles_final.qml`. You should see:
- 37 green poles with completed bases (no more floating half-poles)
- Blue SAME_LANE wires running along each side of the road
- Red CROSS_ROAD wires spanning between the two lanes

Also open `wires.csv` to review which pole pairs got wires and by which pass. If a span you expected to see is missing, its row will be absent — most likely it has no raw lidar returns between the two poles (data limitation, not algorithm).

**Party time 🎉**